<a href="https://colab.research.google.com/github/bsheese/387/blob/main/19_onramp/19_1_ML_Foundations/19_1_1_Is_It_Any_Good.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The On-Ramp: Part 1
## Is It Any Good?

---

## What This Notebook Is About

The pet-breed model from 19_0 got 180 of the 200 test photos right, an accuracy of 0.90. That sounds good, but a single number like that doesn't mean much until you can answer two more questions:

1. **Compared with what?** Is 0.90 hard to get on this problem, or would almost anything get it? To answer that we need a **baseline**, the score of a simple method that involves no real learning.
2. **Measured on which photos?** A model can do much better on the photos it learned from than on new ones. We need to know which photos a score comes from, and why the training, validation and test photos are kept apart.

The four 19_1 notebooks all work with the model you trained in 19_0, using the predictions and features saved at the end of 19_0_2. Nothing here needs a GPU, and every cell runs in a few seconds.

**What you will learn:**
1. How to work with a model's predictions as NumPy arrays, and compute accuracy from them
2. What a baseline is, and three baselines for the pet problem
3. Why a model's score on its own training photos is not a fair measure
4. What overfitting looks like when a model trains for too long
5. Why there is a validation set *and* a test set

**Optional reference.** If you took CS 377, you have seen these ideas before in [16_1](https://github.com/bsheese/cs377/blob/main/16_ml_intro/16_1_Intro_to_ML.ipynb), [17_0_1](https://github.com/bsheese/cs377/blob/main/17_regression_crossval/17_0_Preliminaries/17_0_1_UnivariateSpread.ipynb), [17_1_6](https://github.com/bsheese/cs377/blob/main/17_regression_crossval/17_1_SLR/17_1_6_SLR_GeneralizationTest.ipynb). If you haven't, those notebooks cover the same ideas on other datasets, in more depth. You don't need them for this course.

---

## Section 1: The Model's Predictions as Arrays

The next cell loads the libraries and downloads `pets4_predictions.csv`, the file 19_0_2 saved. It has one row for each of the 800 photos, with the photo's split (train, val or test), its true breed, the model's predicted breed, and the model's scores and probabilities.

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name

preds = pd.read_csv(fetch('pets4_predictions.csv'))
preds[['file', 'split', 'breed', 'predicted', 'prob_Birman', 'prob_Ragdoll', 'prob_basset_hound', 'prob_beagle']].head()

That is a pandas DataFrame, a table with named columns. We use pandas only to read the file. For the calculations we take the columns out as **NumPy arrays**. A NumPy array is like a Python list, except that every element has the same type and you can do an operation on the whole array at once, without writing a loop. Most machine learning code, including PyTorch, works with arrays like these.

In [ ]:
split = preds['split'].to_numpy()
breed = preds['breed'].to_numpy()
predicted = preds['predicted'].to_numpy()

print(type(breed))
print('shape:', breed.shape)
print('first five true breeds:     ', breed[:5])
print('first five predicted breeds:', predicted[:5])

`breed.shape` is `(800,)`: a one-dimensional array with 800 elements. Indexing and slicing work the way they do for lists, so `breed[:5]` is the first five elements.

The useful difference from lists is that comparisons work element by element. `predicted == breed` compares the two arrays position by position and gives an array of 800 `True`/`False` values, `True` wherever the prediction was right.

In [ ]:
right = predicted == breed
print(right[:5])
print('number right:', right.sum())
print('fraction right:', right.mean())

Python counts `True` as 1 and `False` as 0, so `.sum()` counts the `True` values and `.mean()` gives the fraction that are `True`. The fraction of predictions that are right is the model's **accuracy**.

That accuracy mixes all 800 photos together, including the 400 the model learned from. To get the test accuracy we need only the test rows. The same trick works for selecting rows: `split == 'test'` is a `True`/`False` array, and indexing an array with it keeps only the positions that are `True`. An array used this way is called a **mask**.

In [ ]:
is_test = split == 'test'
print('test photos:', is_test.sum())

test_accuracy = (predicted[is_test] == breed[is_test]).mean()
print(f'test accuracy: {test_accuracy:.3f}')

That's the 0.90 from 19_0. Now we can ask whether it is any good.

---

## Section 2: Compared with What? Baselines

A **baseline** is the score you get from a simple method that doesn't learn anything from the photos. If a model can't beat its baseline clearly, the model isn't doing anything useful, however impressive its accuracy sounds. Baselines also tell you how hard a problem is. An accuracy of 0.90 is excellent if the baseline is 0.25 and useless if the baseline is 0.95.

### Baseline 1: always predict the most common breed

The simplest baseline ignores the photo completely and always gives the answer that's right most often in the training photos. `np.unique` with `return_counts=True` lists each distinct value in an array and how many times it appears.

In [ ]:
is_train = split == 'train'
names, counts = np.unique(breed[is_train], return_counts=True)
print(names)
print(counts)

All four breeds appear 100 times in the training photos, so there is no single most common breed. Any one breed will do; `np.argmax` gives the position of the largest count, and with a tie it picks the first one.

In [ ]:
most_common = names[np.argmax(counts)]
print('most common breed:', most_common)

baseline_accuracy = (breed[is_test] == most_common).mean()
print(f'accuracy of always predicting {most_common}: {baseline_accuracy:.3f}')

This baseline gets 0.25 because each breed is exactly a quarter of the test photos. That's only true because the dataset was built with equal numbers of each breed. Real data is rarely balanced like that. If 90 of every 100 photos sent to the model were beagles, always answering "beagle" would score 0.90, the same as our model, without looking at a single photo. So a model's accuracy has to be read next to the baseline for the same data. 19_1_2 comes back to this.

### Baseline 2: guess at random

Another simple method picks one of the four breeds at random for every photo. `np.random.default_rng` makes a random number generator, and its `choice` method picks from a list. Setting the seed (387 here) makes the random choices the same every time the cell runs.

In [ ]:
BREEDS = ['Birman', 'Ragdoll', 'basset_hound', 'beagle']
rng = np.random.default_rng(387)

random_guesses = rng.choice(BREEDS, size=is_test.sum())
print(random_guesses[:5])
print(f'accuracy of random guessing: {(random_guesses == breed[is_test]).mean():.3f}')

Random guessing is right about one time in four, so it also gets close to 0.25. It won't be exactly 0.25, because the guesses happen to land a little differently each time.

### Baseline 3: tell cats from dogs, then guess the breed

Both baselines so far are very weak. A tougher one asks how much of the model's accuracy comes from the easy part of the problem. Suppose a method could always tell a cat from a dog, but then had to guess between the two breeds of that species. That would be right about half the time.

In [ ]:
CATS = ['Birman', 'Ragdoll']
DOGS = ['basset_hound', 'beagle']

species_guesses = []
for b in breed[is_test]:
    if b in CATS:
        species_guesses.append(rng.choice(CATS))
    else:
        species_guesses.append(rng.choice(DOGS))
species_guesses = np.array(species_guesses)

print(f'accuracy of species-then-guess: {(species_guesses == breed[is_test]).mean():.3f}')

(As with random guessing, the random choices make it land a little away from exactly 0.5.)

Our model gets 0.90, well above all three baselines, including the one that already knows cat from dog. So the model has learned something about the differences between look-alike breeds, which is the hard part of the problem.

Picking a good baseline takes some thought. The most common answer is always worth computing, because it's easy and it catches the case where a high accuracy comes only from unbalanced data. A baseline that solves the easy part of the problem, like the species baseline here, tells you more about whether the model learned the part you care about.

---

## Section 3: Measured on Which Photos?

Here is a "model" that gets every training photo right. It looks up the photo's file name in a dictionary of the training photos and returns the breed stored there.

In [ ]:
files = preds['file'].to_numpy()

memory = {}
for f, b in zip(files[is_train], breed[is_train]):
    memory[f] = b

def memorizer(f):
    return memory.get(f, 'no idea')

train_answers = np.array([memorizer(f) for f in files[is_train]])
test_answers = np.array([memorizer(f) for f in files[is_test]])
print(f'memorizer on training photos: {(train_answers == breed[is_train]).mean():.3f}')
print(f'memorizer on test photos:     {(test_answers == breed[is_test]).mean():.3f}')
print('its answers on the first three test photos:', test_answers[:3])

The memorizer is perfect on the photos it has seen and useless on everything else. A real model is never this extreme, but it can drift in the same direction: it can learn details of the particular training photos (this cat's blanket, that dog's collar) that don't help with new photos. The training photos are the ones it was rewarded for getting right, so its score on them will tend to be too high.

That's why the score that matters is the score on photos the model never learned from. Here is our model's accuracy on each of the three groups of photos.

In [ ]:
for s in ['train', 'val', 'test']:
    mask = split == s
    print(f'{s:5}  accuracy {(predicted[mask] == breed[mask]).mean():.3f}   ({mask.sum()} photos)')

The training accuracy is a little higher than the validation and test accuracies, which are close to each other. A small gap like this is normal. A large gap, with high training accuracy and much lower accuracy on new photos, means the model has learned things about its training photos that don't carry over. That is called **overfitting**.

---

## Section 4: Training Too Long

In 19_0_1 the model trained for 8 epochs. What happens if it trains for much longer?

Retraining the whole network would take a long time, but we don't need to. In 19_0_2 we saw that the frozen part of ResNet-18 turns every photo into 512 features, and only the last layer is trained. 19_0_2 saved those features, so we can train a fresh last layer directly on them and skip running the photos through the network. Each epoch then takes a fraction of a second, because 400 rows of 512 numbers is a small amount of data.

The next cell loads the features. `features` is a two-dimensional array with one row per photo and one column per feature, in the same order as the rows of `preds`.

In [ ]:
saved = np.load(fetch('pets4_features.npz'))
features = saved['features']
print('features:', features.shape)
print('same row order as preds:', (saved['filename'] == files).all())

The training code below is the same loop as in 19_0_1, with two simplifications. The model is just the last layer, `nn.Linear(512, 4)`, and each epoch uses all 400 training photos at once instead of batches of 32. After every epoch it records the accuracy and the loss on the training photos and on the validation photos. (The **loss** measures how wrong the model's probabilities are. It's lower when the model gives a high probability to the right breed. Unit 20 explains it.) Unit 21 explains every line of this loop.

In [ ]:
labels = np.array([BREEDS.index(b) for b in breed])   # breed names as numbers 0-3
X_train = torch.tensor(features[is_train])
y_train = torch.tensor(labels[is_train])
is_val = split == 'val'
X_val = torch.tensor(features[is_val])
y_val = torch.tensor(labels[is_val])

torch.manual_seed(0)
last_layer = nn.Linear(512, 4)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(last_layer.parameters(), lr=0.003)

EPOCHS = 300
history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
for epoch in range(EPOCHS):
    loss = loss_fn(last_layer(X_train), y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        for name, X, y in [('train', X_train, y_train), ('val', X_val, y_val)]:
            scores = last_layer(X)
            history[f'{name}_loss'].append(loss_fn(scores, y).item())
            history[f'{name}_acc'].append((scores.argmax(dim=1) == y).float().mean().item())

print(f"after {EPOCHS} epochs: training accuracy {history['train_acc'][-1]:.3f}, validation accuracy {history['val_acc'][-1]:.3f}")

In [ ]:
epochs = np.arange(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history['train_acc'], label='training photos')
ax1.plot(epochs, history['val_acc'], label='validation photos')
ax1.set_xlabel('epoch')
ax1.set_ylabel('accuracy')
ax1.set_ylim(0.4, 1.02)
ax1.set_title('Accuracy')
ax1.legend()

ax2.plot(epochs, history['train_loss'], label='training photos')
ax2.plot(epochs, history['val_loss'], label='validation photos')
ax2.set_xlabel('epoch')
ax2.set_ylabel('loss (lower is better)')
ax2.set_ylim(0, 1)
ax2.set_title('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

Read the two plots from left to right.

- **Early on**, both accuracies rise and both losses fall. The model is learning things that help with every photo, seen or unseen.
- **After that**, the training accuracy keeps climbing until it reaches 1.0, and the training loss keeps falling toward 0. The validation accuracy stops improving at around 0.9, and the validation loss stops falling and starts to rise again.

The model keeps getting better on the photos it trains on and stops getting better (by the loss, it gets slightly worse) on photos it hasn't seen. From that point on, what it learns is specific to the 400 training photos. That is overfitting. If we only looked at the training photos, we'd conclude the model was perfect.

The overfitting here is mild, because only the 2,052 numbers in the last layer are being trained, on top of features that were learned from a million other photos. Unit 21 trains whole networks, where the gap between the training and validation curves can be much larger.

The validation curve tells us when to stop: around the epoch where the validation loss is lowest.

In [ ]:
best_epoch = int(np.argmin(history['val_loss'])) + 1
print('epoch with the lowest validation loss:', best_epoch)
print(f"validation accuracy at that epoch: {history['val_acc'][best_epoch - 1]:.3f}")

---

## Section 5: Why Not Just Use the Test Photos?

We just used the validation photos to make a choice: how many epochs to train for. In practice we make many choices this way, such as which breeds to train on, how many photos to use and how fast the model learns. Each time, we try several options and keep whichever one did best on the validation photos.

Picking the best of several options on a set of photos makes the winner's score on those photos optimistic. Some of the winner's advantage is real, and some is luck: those particular validation photos happened to suit it.

The next cell shows the effect. It trains 30 different versions of the last layer, each on a different random 10 training photos per breed, the way you might compare 30 options. It records each version's accuracy on the validation photos and on the test photos, then picks the version with the best validation accuracy.

In [ ]:
X_test = torch.tensor(features[is_test])
y_test = torch.tensor(labels[is_test])
train_positions = np.where(is_train)[0]
rng = np.random.default_rng(0)

def accuracy(layer, X, y):
    with torch.no_grad():
        return (layer(X).argmax(dim=1) == y).float().mean().item()

val_acc, test_acc = [], []
for version in range(30):
    # a different random 10 training photos of each breed for every version
    chosen = np.concatenate([rng.choice(train_positions[labels[is_train] == k], 10, replace=False)
                             for k in range(4)])
    X, y = torch.tensor(features[chosen]), torch.tensor(labels[chosen])
    torch.manual_seed(version)
    layer = nn.Linear(512, 4)
    optimizer = torch.optim.Adam(layer.parameters(), lr=0.01)
    for epoch in range(50):
        loss = loss_fn(layer(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    val_acc.append(accuracy(layer, X_val, y_val))
    test_acc.append(accuracy(layer, X_test, y_test))

val_acc, test_acc = np.array(val_acc), np.array(test_acc)
winner = np.argmax(val_acc)
print(f'                 validation   test')
print(f'average version     {val_acc.mean():.3f}     {test_acc.mean():.3f}')
print(f'winning version     {val_acc[winner]:.3f}     {test_acc[winner]:.3f}')
print(f'winner - average   {val_acc[winner] - val_acc.mean():+.3f}    {test_acc[winner] - test_acc.mean():+.3f}')

Compare each set of photos with itself. On the validation photos, the winning version looks about 4 percentage points better than the average version. On the test photos, which played no part in the choice, it is barely better than average. Most of the winner's advantage on the validation photos was luck that didn't carry over to new photos. (The validation and test photos differ a little in difficulty, which is why the averages in the two columns aren't equal. That's why we compare the winner with the average in each column.)

So the winner's validation score overstates how good it is, and only the test photos, which weren't used to choose it, give an honest estimate.

That's the reason for keeping three separate groups of photos:

| Photos | Used for | Rule |
|---|---|---|
| **Training** | learning the weights | the model's score on them is too high |
| **Validation** | comparing options and making choices (epochs, settings, which model) | the best option's score on them is a little too high |
| **Test** | one final measurement after every choice is made | look at it once; if you change the model because of it, it stops being a test |

If we went back and changed the model after seeing the test score, the test photos would become one more set of validation photos, and we'd need fresh photos to get an honest score.

---

## Summary

- A model's predictions can be handled as NumPy arrays. `predicted == breed` gives a `True`/`False` array, `.mean()` of it is the accuracy, and a `True`/`False` array used as an index (a **mask**) selects rows, such as the test photos.
- A **baseline** is the score of a simple method that doesn't learn from the data. Always predicting the most common breed and guessing at random both get about 0.25 here; telling cats from dogs and then guessing gets about 0.5. The model's 0.90 is far above all three.
- Accuracy only means something next to the baseline for the same data. With unbalanced data, the most-common-answer baseline can be very high.
- A model's score on its own training photos is too high. In the extreme case, a memorizer gets every training photo right and can't answer anything else.
- **Overfitting**: as training goes on, training accuracy keeps rising while validation accuracy stops improving and validation loss starts to rise.
- The **validation** photos are for making choices, which makes the chosen option's validation score a bit optimistic. The **test** photos are kept for one final, honest measurement.

**Next:** 19_1_2 breaks the 0.90 down breed by breed with a confusion matrix, and shows a case where accuracy hides a model that is no use at all.